# Time To Reversion (TTR) Analysis

## Hypothesis

**RSI(14) > SMA(RSI(14), 14)** represents positive MA acceleration (bullish bias).  
**RSI(14) < SMA(RSI(14), 14)** represents negative MA acceleration (bearish bias).

The claim is:
- When RSI(14) **>** its own SMA(14) (bullish acceleration), an RSI(2) oversold signal → **faster** mean reversion (shorter TTR)
- When RSI(14) **<** its own SMA(14) (bearish acceleration), an RSI(2) overbought signal → **faster** mean reversion (shorter TTR)

We define **Time To Reversion (TTR)** as the number of bars for RSI(2) to cross back above a threshold after an oversold/overbought extreme.

We'll test this statistically with:
1. Mann-Whitney U test (non-parametric, robust to non-normal distributions)
2. Cohen's d (effect size)
3. ECDF visualization

In [2]:
import sys
import os
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import mannwhitneyu

warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.bybit import BybitConnector
from utils.indicators import get_wilders_rsi, sma

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print("\u2713 Imports loaded")

✓ Imports loaded


In [3]:
# ══════════════════════════════════════════════════════════════════════
# SECTION 1 — DATA COLLECTION
# ══════════════════════════════════════════════════════════════════════

print("=" * 72)
print("  TTR ANALYSIS — TIME TO REVERSION")
print("=" * 72)

print("\n[1/4] Fetching BTCUSDT 1-min OHLCV data...")
client = BybitConnector(api_key="", api_secret="", testnet=False)
price = client.get_ohlcv("BTC/USDT:USDT", "1", 10000)
close = np.asarray(price['close'], dtype=float)
high = np.asarray(price['high'], dtype=float)
low = np.asarray(price['low'], dtype=float)
print(f"  \u2713 Fetched {len(close)} bars ({price['timestamp'].iloc[0].strftime('%Y-%m-%d')} to {price['timestamp'].iloc[-1].strftime('%Y-%m-%d')})")

  TTR ANALYSIS — TIME TO REVERSION

[1/4] Fetching BTCUSDT 1-min OHLCV data...
  ✓ Fetched 9999 bars (2026-07-14 to 2026-07-21)


In [4]:
# ══════════════════════════════════════════════════════════════════════
# SECTION 2 — CALCULATE INDICATORS
# ══════════════════════════════════════════════════════════════════════

print("\n[2/4] Computing indicators...")

# RSI(14) — the slow/trend RSI
delta = np.diff(close, prepend=close[0])
rsi_14 = get_wilders_rsi(delta, 14)

# SMA of RSI(14) — the smoothed baseline
rsi_14_sma = sma(rsi_14, 14)

# RSI(2) — the fast mean-reversion RSI
rsi_2 = get_wilders_rsi(delta, 2)

# Align arrays (sma is shorter by window-1 due to padding)
align_sma = len(rsi_14) - len(rsi_14_sma)
rsi_14_sma_padded = np.concatenate([np.full(align_sma, np.nan), rsi_14_sma])

print(f"  \u2713 RSI(14) computed: {len(rsi_14)} values")
print(f"  \u2713 SMA(RSI(14),14) computed: {len(rsi_14_sma_padded)} values")
print(f"  \u2713 RSI(2) computed: {len(rsi_2)} values")


[2/4] Computing indicators...
  ✓ RSI(14) computed: 9999 values
  ✓ SMA(RSI(14),14) computed: 9999 values
  ✓ RSI(2) computed: 9999 values


In [5]:
# ══════════════════════════════════════════════════════════════════════
# SECTION 3 — DETECT TTR EVENTS
# ══════════════════════════════════════════════════════════════════════

print("\n[3/4] Detecting TTR events...")

# Build DataFrame for alignment
df = pd.DataFrame({
    'rsi_14': rsi_14,
    'rsi_14_sma': rsi_14_sma_padded,
    'rsi_2': rsi_2,
})
df = df.dropna().reset_index(drop=True)

# Regime: True if RSI(14) > SMA(RSI(14), 14) — bullish acceleration
#          False if RSI(14) < SMA(RSI(14), 14) — bearish acceleration
df['regime_bull'] = df['rsi_14'] > df['rsi_14_sma']

# ── OVERSOLD + BULLISH REGIME ──────────────────────────────────────────
# RSI(2) < 30 (oversold) while RSI(14) > SMA(14) (bullish accel)
# Expectation: fast reversion

# ── OVERSOLD + BEARISH REGIME ──────────────────────────────────────────
# RSI(2) < 30 (oversold) while RSI(14) < SMA(14) (bearish accel)
# Expectation: slow reversion (might stay oversold longer)

# ── OVERBOUGHT + BEARISH REGIME ────────────────────────────────────────
# RSI(2) > 70 (overbought) while RSI(14) < SMA(14) (bearish accel)
# Expectation: fast reversion

# ── OVERBOUGHT + BULLISH REGIME ────────────────────────────────────────
# RSI(2) > 70 (overbought) while RSI(14) > SMA(14) (bullish accel)
# Expectation: slow reversion


MAX_LOOKAHEAD = 200  # Max bars to look ahead for reversion

def detect_ttr_events(df, rsi_col, threshold, direction, regime_col, regime_value, max_lookahead=MAX_LOOKAHEAD):
    """
    Detect TTR events.
    
    Parameters:
    - df: DataFrame with indicator values
    - rsi_col: column name for the RSI to use (e.g., 'rsi_2')
    - threshold: e.g., 30 for oversold, 70 for overbought
    - direction: 'below' (extreme then revert up) or 'above' (extreme then revert down)
    - regime_col: column name for regime boolean
    - regime_value: True or False to filter regime
    """
    rsi = df[rsi_col].values
    regime = df[regime_col].values
    
    ttr_values = []
    event_indices = []
    
    i = 0
    while i < len(df) - 5:
        # Check regime condition at this bar
        if regime[i] != regime_value:
            i += 1
            continue
        
        # Check if RSI is in extreme territory
        if direction == 'below' and rsi[i] < threshold:
            pass  # oversold
        elif direction == 'above' and rsi[i] > threshold:
            pass  # overbought
        else:
            i += 1
            continue
        
        # Now look forward for reversion
        lookahead = min(max_lookahead, len(df) - i - 1)
        found = False
        
        if direction == 'below':
            # Reversion means RSI crosses back ABOVE the threshold
            # We also require it crosses above 50 (neutral) as stronger confirmation
            revert_target = 50  # reversion to neutral
            for j in range(1, lookahead):
                if rsi[i + j] >= revert_target:
                    ttr_values.append(j)
                    event_indices.append(i)
                    found = True
                    i += j  # Skip ahead to avoid overlapping events
                    break
        else:  # direction == 'above'
            revert_target = 50  # reversion to neutral
            for j in range(1, lookahead):
                if rsi[i + j] <= revert_target:
                    ttr_values.append(j)
                    event_indices.append(i)
                    found = True
                    i += j
                    break
        
        if not found:
            i += 1
    
    return np.array(ttr_values), event_indices


# ── Detect oversold events (RSI(2) < 30) ──────────────────────────────
ttr_os_bull, idx_os_bull = detect_ttr_events(
    df, 'rsi_2', 30, 'below', 'regime_bull', True
)
ttr_os_bear, idx_os_bear = detect_ttr_events(
    df, 'rsi_2', 30, 'below', 'regime_bull', False
)

# ── Detect overbought events (RSI(2) > 70) ────────────────────────────
ttr_ob_bull, idx_ob_bull = detect_ttr_events(
    df, 'rsi_2', 70, 'above', 'regime_bull', True
)
ttr_ob_bear, idx_ob_bear = detect_ttr_events(
    df, 'rsi_2', 70, 'above', 'regime_bull', False
)

print(f"\n  Oversold (RSI(2) < 30) events:")
print(f"    Bullish regime (RSI14 > SMA14): {len(ttr_os_bull)} events")
print(f"    Bearish regime (RSI14 < SMA14): {len(ttr_os_bear)} events")
print(f"\n  Overbought (RSI(2) > 70) events:")
print(f"    Bullish regime (RSI14 > SMA14): {len(ttr_ob_bull)} events")
print(f"    Bearish regime (RSI14 < SMA14): {len(ttr_ob_bear)} events")


[3/4] Detecting TTR events...

  Oversold (RSI(2) < 30) events:
    Bullish regime (RSI14 > SMA14): 240 events
    Bearish regime (RSI14 < SMA14): 1067 events

  Overbought (RSI(2) > 70) events:
    Bullish regime (RSI14 > SMA14): 1002 events
    Bearish regime (RSI14 < SMA14): 261 events


In [6]:
# ══════════════════════════════════════════════════════════════════════
# SECTION 4 — STATISTICAL TESTS
# ══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SECTION 4 — STATISTICAL TESTS")
print("=" * 72)

def cohens_d(x, y):
    """Cohen's d effect size (pooled std)."""
    nx, ny = len(x), len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(
        ((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / dof
    )
    return (np.mean(x) - np.mean(y)) / pooled_std if pooled_std != 0 else 0.0


def run_test(name, group_a, group_b, label_a, label_b):
    """Run Mann-Whitney U test and print results."""
    print(f"\n--- {name} ---")
    print(f"  {label_a}: n={len(group_a):5d}, mean TTR={np.mean(group_a):7.2f} bars, median={np.median(group_a):7.2f}, std={np.std(group_a):.2f}")
    print(f"  {label_b}: n={len(group_b):5d}, mean TTR={np.mean(group_b):7.2f} bars, median={np.median(group_b):7.2f}, std={np.std(group_b):.2f}")
    
    if len(group_a) > 0 and len(group_b) > 0:
        stat, pval = mannwhitneyu(group_a, group_b, alternative='two-sided')
        d = cohens_d(group_a, group_b)
        print(f"  Mann-Whitney U: stat={stat:.1f}, p-value={pval:.6f}")
        print(f"  Cohen's d: {d:.4f}")
        
        # Effect size interpretation
        if abs(d) < 0.2:
            eff_str = "negligible"
        elif abs(d) < 0.5:
            eff_str = "small"
        elif abs(d) < 0.8:
            eff_str = "medium"
        else:
            eff_str = "large"
        print(f"  Effect size: {eff_str}")
        
        if pval < 0.05:
            if np.mean(group_a) < np.mean(group_b):
                print(f"  \u2713 CONFIRMED: {label_a} reverts FASTER (shorter TTR) \u2014 p={pval:.6f}")
            else:
                print(f"  \u2717 CONTRADICTED: {label_b} reverts faster despite hypothesis")
        else:
            print(f"  \u2753 Not statistically significant (p={pval:.4f} \u2265 0.05)")
    else:
        print("  \u26A0 Insufficient events in one or both groups.")

    return stat, float(pval) if len(group_a) > 0 and len(group_b) > 0 else 1.0, d


# ── HYPOTHESIS 1: Oversold + Bullish Regime → faster reversion ────────
# Expectation: TTR_os_bull < TTR_os_bear
print("\n" + "-" * 72)
print("  HYPOTHESIS 1: OVERSOLD (RSI2 < 30) — REVERSION TO 50")
print("  Claim: Bullish regime (RSI14 > SMA14) → FASTER reversion")
print("-" * 72)
h1_stat, h1_pval, h1_d = run_test(
    "Oversold: Bullish vs Bearish Regime",
    ttr_os_bull, ttr_os_bear,
    "Bullish (RSI14>SMA14)", "Bearish (RSI14<SMA14)"
)

# ── HYPOTHESIS 2: Overbought + Bearish Regime → faster reversion ──────
# Expectation: TTR_ob_bear < TTR_ob_bull
print("\n" + "-" * 72)
print("  HYPOTHESIS 2: OVERBOUGHT (RSI2 > 70) — REVERSION TO 50")
print("  Claim: Bearish regime (RSI14 < SMA14) → FASTER reversion")
print("-" * 72)
h2_stat, h2_pval, h2_d = run_test(
    "Overbought: Bearish vs Bullish Regime",
    ttr_ob_bear, ttr_ob_bull,
    "Bearish (RSI14<SMA14)", "Bullish (RSI14>SMA14)"
)


  SECTION 4 — STATISTICAL TESTS

------------------------------------------------------------------------
  HYPOTHESIS 1: OVERSOLD (RSI2 < 30) — REVERSION TO 50
  Claim: Bullish regime (RSI14 > SMA14) → FASTER reversion
------------------------------------------------------------------------

--- Oversold: Bullish vs Bearish Regime ---
  Bullish (RSI14>SMA14): n=  240, mean TTR=   3.34 bars, median=   2.50, std=2.82
  Bearish (RSI14<SMA14): n= 1067, mean TTR=   3.37 bars, median=   2.00, std=2.80
  Mann-Whitney U: stat=129222.0, p-value=0.819560
  Cohen's d: -0.0092
  Effect size: negligible
  ❓ Not statistically significant (p=0.8196 ≥ 0.05)

------------------------------------------------------------------------
  HYPOTHESIS 2: OVERBOUGHT (RSI2 > 70) — REVERSION TO 50
  Claim: Bearish regime (RSI14 < SMA14) → FASTER reversion
------------------------------------------------------------------------

--- Overbought: Bearish vs Bullish Regime ---
  Bearish (RSI14<SMA14): n=  261, mean

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SECTION 5 — ROBUSTNESS CHECKS
# ══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SECTION 5 \u2014 ROBUSTNESS CHECKS")
print("=" * 72)

# ── 5a. Different reversion thresholds ──────────────────────────────────
print("\n--- 5a. Alternative Reversion Thresholds ---")

def test_alt_thresholds(df, extreme_threshold, extreme_dir, max_lookahead=200):
    """Test TTR for different reversion targets."""
    results = {}
    for revert_target in [30, 40, 50, 60, 70]:
        if (extreme_dir == 'below' and revert_target <= extreme_threshold) or \
           (extreme_dir == 'above' and revert_target >= extreme_threshold):
            continue
        
        def detect_with_target(df, regime_val, revert_target):
            rsi = df['rsi_2'].values
            regime = df['regime_bull'].values
            ttr_vals = []
            i = 0
            while i < len(df) - 5:
                if regime[i] != regime_val:
                    i += 1
                    continue
                
                if extreme_dir == 'below' and rsi[i] < extreme_threshold:
                    pass
                elif extreme_dir == 'above' and rsi[i] > extreme_threshold:
                    pass
                else:
                    i += 1
                    continue
                
                lookahead = min(max_lookahead, len(df) - i - 1)
                found = False
                for j in range(1, lookahead):
                    if (extreme_dir == 'below' and rsi[i + j] >= revert_target) or \
                       (extreme_dir == 'above' and rsi[i + j] <= revert_target):
                        ttr_vals.append(j)
                        found = True
                        i += j
                        break
                if not found:
                    i += 1
            return np.array(ttr_vals)
        
        bull_ttr = detect_with_target(df, True, revert_target)
        bear_ttr = detect_with_target(df, False, revert_target)
        
        if len(bull_ttr) > 0 and len(bear_ttr) > 0:
            stat, pval = mannwhitneyu(bull_ttr, bear_ttr, alternative='two-sided')
            d = cohens_d(bull_ttr, bear_ttr)
            direction = "bull_faster" if np.mean(bull_ttr) < np.mean(bear_ttr) else "bear_faster"
            results[f"revert_to_{revert_target}"] = {
                "n_bull": len(bull_ttr), "mean_bull": np.mean(bull_ttr),
                "n_bear": len(bear_ttr), "mean_bear": np.mean(bear_ttr),
                "pval": pval, "d": d, "direction": direction
            }
    return results

# Test oversold with alternative thresholds
print("  Oversold (RSI2 < 30) \u2014 alt reversion targets:")
os_alt = test_alt_thresholds(df, 30, 'below')
for k, v in os_alt.items():
    sig = "***" if v['pval'] < 0.001 else "**" if v['pval'] < 0.01 else "*" if v['pval'] < 0.05 else ""
    faster = "BULL" if v['direction'] == 'bull_faster' else "BEAR"
    print(f"    {k}: bull_mean={v['mean_bull']:.1f}(n={v['n_bull']}), bear_mean={v['mean_bear']:.1f}(n={v['n_bear']}), "
          f"p={v['pval']:.4f}{sig}, d={v['d']:.3f}, faster={faster}")

print("  Overbought (RSI2 > 70) \u2014 alt reversion targets:")
ob_alt = test_alt_thresholds(df, 70, 'above')
for k, v in ob_alt.items():
    sig = "***" if v['pval'] < 0.001 else "**" if v['pval'] < 0.01 else "*" if v['pval'] < 0.05 else ""
    faster = "BULL" if v['direction'] == 'bull_faster' else "BEAR"
    print(f"    {k}: bull_mean={v['mean_bull']:.1f}(n={v['n_bull']}), bear_mean={v['mean_bear']:.1f}(n={v['n_bear']}), "
          f"p={v['pval']:.4f}{sig}, d={v['d']:.3f}, faster={faster}")

# ── 5b. Using RSI(14) - SMA(RSI14) magnitude instead of binary regime ──
print("\n--- 5b. Distance-from-SMA magnitude (continuous analysis) ---")

df['rsi_sma_dist'] = df['rsi_14'] - df['rsi_14_sma']

# For oversold events, does a larger positive distance correlate with shorter TTR?
oversold_mask = df['rsi_2'] < 30
# Get unique oversold events with their distance value
os_events = []
i = 0
while i < len(df):
    if oversold_mask.iloc[i]:
        dist = df['rsi_sma_dist'].iloc[i]
        # Find reversion
        for j in range(1, 200):
            if i + j >= len(df):
                break
            if df['rsi_2'].iloc[i + j] >= 50:
                os_events.append({'ttr': j, 'dist': dist, 'regime': df['regime_bull'].iloc[i]})
                i += j
                break
        else:
            i += 1
    else:
        i += 1

os_events_df = pd.DataFrame(os_events)
if len(os_events_df) > 0:
    # Correlation between distance and TTR
    r, p_corr = stats.pearsonr(os_events_df['dist'], os_events_df['ttr'])
    rho, p_spearman = stats.spearmanr(os_events_df['dist'], os_events_df['ttr'])
    check_mark = "\u2713"
    print(f"  Oversold: Pearson r={r:.4f} (p={p_corr:.4f}), Spearman \u03c1={rho:.4f} (p={p_spearman:.4f})")
    print(f"    Interpretation: {'Distance correlates with shorter TTR ' + check_mark if r < 0 and p_corr < 0.05 else 'No significant linear relationship'}")

# For overbought events
overbought_mask = df['rsi_2'] > 70
ob_events = []
i = 0
while i < len(df):
    if overbought_mask.iloc[i]:
        dist = df['rsi_sma_dist'].iloc[i]
        for j in range(1, 200):
            if i + j >= len(df):
                break
            if df['rsi_2'].iloc[i + j] <= 50:
                ob_events.append({'ttr': j, 'dist': dist, 'regime': df['regime_bull'].iloc[i]})
                i += j
                break
        else:
            i += 1
    else:
        i += 1

ob_events_df = pd.DataFrame(ob_events)
if len(ob_events_df) > 0:
    # For overbought, a NEGATIVE distance (bearish) should correlate with shorter TTR
    r, p_corr = stats.pearsonr(ob_events_df['dist'], ob_events_df['ttr'])
    rho, p_spearman = stats.spearmanr(ob_events_df['dist'], ob_events_df['ttr'])
    check_mark = "\u2713"
    print(f"  Overbought: Pearson r={r:.4f} (p={p_corr:.4f}), Spearman \u03c1={rho:.4f} (p={p_spearman:.4f})")
    print(f"    Interpretation: {'Distance correlates with shorter TTR ' + check_mark + ' (neg dist = faster)' if r > 0 and p_corr < 0.05 else 'No significant linear relationship'}")


  SECTION 5 — ROBUSTNESS CHECKS

--- 5a. Alternative Reversion Thresholds ---
  Oversold (RSI2 < 30) — alt reversion targets:
    revert_to_40: bull_mean=2.7(n=246), bear_mean=2.8(n=1168), p=0.6238, d=-0.061, faster=BULL
    revert_to_50: bull_mean=3.3(n=240), bear_mean=3.4(n=1067), p=0.8196, d=-0.009, faster=BULL
    revert_to_60: bull_mean=4.0(n=239), bear_mean=4.2(n=946), p=0.2962, d=-0.074, faster=BULL
    revert_to_70: bull_mean=5.3(n=237), bear_mean=5.8(n=801), p=0.0465*, d=-0.116, faster=BULL
  Overbought (RSI2 > 70) — alt reversion targets:
    revert_to_30: bull_mean=5.6(n=786), bear_mean=4.9(n=252), p=0.0018**, d=0.161, faster=BEAR
    revert_to_40: bull_mean=4.2(n=903), bear_mean=3.7(n=258), p=0.0040**, d=0.166, faster=BEAR
    revert_to_50: bull_mean=3.4(n=1002), bear_mean=3.0(n=261), p=0.0039**, d=0.136, faster=BEAR
    revert_to_60: bull_mean=2.9(n=1088), bear_mean=2.7(n=266), p=0.0057**, d=0.096, faster=BEAR

--- 5b. Distance-from-SMA magnitude (continuous analysis) ---

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SECTION 6 — VISUALIZATION
# ══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SECTION 6 \u2014 VISUALIZATION")
print("=" * 72)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Time To Reversion (TTR) Analysis — RSI(2) Extreme Events',
             fontsize=15, fontweight='bold', y=1.02)

# ── Panel 1: Oversold TTR histograms ──────────────────────────────────
ax1 = axes[0, 0]
if len(ttr_os_bull) > 0:
    bins = np.linspace(0, max(ttr_os_bull.max(), ttr_os_bear.max()), 40)
    ax1.hist(ttr_os_bull, bins=bins, alpha=0.6, label=f'Bull regime (n={len(ttr_os_bull)})', color='green')
    ax1.axvline(np.median(ttr_os_bull), color='green', ls='--', lw=2,
                label=f'Median={np.median(ttr_os_bull):.1f}')
if len(ttr_os_bear) > 0:
    ax1.hist(ttr_os_bear, bins=bins, alpha=0.6, label=f'Bear regime (n={len(ttr_os_bear)})', color='red')
    ax1.axvline(np.median(ttr_os_bear), color='red', ls='-.', lw=2,
                label=f'Median={np.median(ttr_os_bear):.1f}')
ax1.set_title('Oversold (RSI2 < 30) — TTR to 50', fontsize=12)
ax1.set_xlabel('Bars to reversion')
ax1.set_ylabel('Frequency')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# ── Panel 2: Overbought TTR histograms ────────────────────────────────
ax2 = axes[0, 1]
if len(ttr_ob_bull) > 0:
    bins = np.linspace(0, max(ttr_ob_bull.max(), ttr_ob_bear.max()), 40)
    ax2.hist(ttr_ob_bull, bins=bins, alpha=0.6, label=f'Bull regime (n={len(ttr_ob_bull)})', color='green')
    ax2.axvline(np.median(ttr_ob_bull), color='green', ls='--', lw=2,
                label=f'Median={np.median(ttr_ob_bull):.1f}')
if len(ttr_ob_bear) > 0:
    ax2.hist(ttr_ob_bear, bins=bins, alpha=0.6, label=f'Bear regime (n={len(ttr_ob_bear)})', color='red')
    ax2.axvline(np.median(ttr_ob_bear), color='red', ls='-.', lw=2,
                label=f'Median={np.median(ttr_ob_bear):.1f}')
ax2.set_title('Overbought (RSI2 > 70) — TTR to 50', fontsize=12)
ax2.set_xlabel('Bars to reversion')
ax2.set_ylabel('Frequency')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ── Panel 3: ECDF Oversold ────────────────────────────────────────────
ax3 = axes[1, 0]
if len(ttr_os_bull) > 0:
    sorted_bull = np.sort(ttr_os_bull)
    ecdf_bull = np.arange(1, len(sorted_bull) + 1) / len(sorted_bull)
    ax3.plot(sorted_bull, ecdf_bull, label=f'Bull regime', color='green', lw=2)
if len(ttr_os_bear) > 0:
    sorted_bear = np.sort(ttr_os_bear)
    ecdf_bear = np.arange(1, len(sorted_bear) + 1) / len(sorted_bear)
    ax3.plot(sorted_bear, ecdf_bear, label=f'Bear regime', color='red', lw=2)
ax3.axhline(0.5, color='gray', ls=':', alpha=0.5, label='50% reversion level')
ax3.set_title('ECDF — Oversold TTR', fontsize=12)
ax3.set_xlabel('Bars to reversion')
ax3.set_ylabel('Cumulative probability')
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)

# Annotate 50th percentile
if len(ttr_os_bull) > 0:
    p50_bull = np.percentile(ttr_os_bull, 50)
    ax3.axvline(p50_bull, color='green', ls=':', alpha=0.4, ymax=0.5)
if len(ttr_os_bear) > 0:
    p50_bear = np.percentile(ttr_os_bear, 50)
    ax3.axvline(p50_bear, color='red', ls=':', alpha=0.4, ymax=0.5)

# ── Panel 4: ECDF Overbought ──────────────────────────────────────────
ax4 = axes[1, 1]
if len(ttr_ob_bull) > 0:
    sorted_bull = np.sort(ttr_ob_bull)
    ecdf_bull = np.arange(1, len(sorted_bull) + 1) / len(sorted_bull)
    ax4.plot(sorted_bull, ecdf_bull, label=f'Bull regime', color='green', lw=2)
if len(ttr_ob_bear) > 0:
    sorted_bear = np.sort(ttr_ob_bear)
    ecdf_bear = np.arange(1, len(sorted_bear) + 1) / len(sorted_bear)
    ax4.plot(sorted_bear, ecdf_bear, label=f'Bear regime', color='red', lw=2)
ax4.axhline(0.5, color='gray', ls=':', alpha=0.5, label='50% reversion level')
ax4.set_title('ECDF — Overbought TTR', fontsize=12)
ax4.set_xlabel('Bars to reversion')
ax4.set_ylabel('Cumulative probability')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)

if len(ttr_ob_bull) > 0:
    p50_bull = np.percentile(ttr_ob_bull, 50)
    ax4.axvline(p50_bull, color='green', ls=':', alpha=0.4, ymax=0.5)
if len(ttr_ob_bear) > 0:
    p50_bear = np.percentile(ttr_ob_bear, 50)
    ax4.axvline(p50_bear, color='red', ls=':', alpha=0.4, ymax=0.5)

plt.tight_layout()
plt.savefig('research/image/ttr_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("  \u2713 Saved to research/image/ttr_analysis.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# SECTION 7 — SUMMARY & CONCLUSION
# ══════════════════════════════════════════════════════════════════════

print("\n" + "=" * 72)
print("  SUMMARY & CONCLUSION")
print("=" * 72)

print("\n" + "-" * 72)
print("  HYPOTHESIS TESTED:")
print("-" * 72)
print("""
  H1: RSI(2) oversold (RSI2 < 30) → faster mean reversion
      when RSI(14) > SMA(RSI14, 14) (bullish acceleration)

  H2: RSI(2) overbought (RSI2 > 70) → faster mean reversion
      when RSI(14) < SMA(RSI14, 14) (bearish acceleration)
""")

print("\n" + "-" * 72)
print("  RESULTS:")
print("-" * 72)

# H1 result
h1_sig = h1_pval < 0.05
h1_correct = np.mean(ttr_os_bull) < np.mean(ttr_os_bear) if (len(ttr_os_bull) > 0 and len(ttr_os_bear) > 0) else False
if h1_sig and h1_correct:
    h1_str = f"\u2713 CONFIRMED (p={h1_pval:.6f}, d={h1_d:.3f})"
elif h1_sig and not h1_correct:
    h1_str = f"\u2717 CONTRADICTED (p={h1_pval:.6f}, opposite direction)"
else:
    h1_str = f"\u2753 NOT SIGNIFICANT (p={h1_pval:.4f})"

# H2 result
h2_sig = h2_pval < 0.05
h2_correct = np.mean(ttr_ob_bear) < np.mean(ttr_ob_bull) if (len(ttr_ob_bull) > 0 and len(ttr_ob_bear) > 0) else False
if h2_sig and h2_correct:
    h2_str = f"\u2713 CONFIRMED (p={h2_pval:.6f}, d={h2_d:.3f})"
elif h2_sig and not h2_correct:
    h2_str = f"\u2717 CONTRADICTED (p={h2_pval:.6f}, opposite direction)"
else:
    h2_str = f"\u2753 NOT SIGNIFICANT (p={h2_pval:.4f})"

print(f"""
  H1 (Oversold + Bull → faster): {h1_str}
  H2 (Overbought + Bear → faster): {h2_str}
""")

# Print summary statistics
print("\n  DETAILED STATISTICS:")
print(f"    {'':>25} {'n':>6} {'Mean TTR':>9} {'Median':>8} {'Std':>7}")
print(f"    {'-'*55}")
if len(ttr_os_bull) > 0:
    print(f"    {'Oversold + Bull regime':>25} {len(ttr_os_bull):>6} {np.mean(ttr_os_bull):>8.1f} {np.median(ttr_os_bull):>7.1f} {np.std(ttr_os_bull):>7.1f}")
if len(ttr_os_bear) > 0:
    print(f"    {'Oversold + Bear regime':>25} {len(ttr_os_bear):>6} {np.mean(ttr_os_bear):>8.1f} {np.median(ttr_os_bear):>7.1f} {np.std(ttr_os_bear):>7.1f}")
if len(ttr_ob_bull) > 0:
    print(f"    {'Overbought + Bull regime':>25} {len(ttr_ob_bull):>6} {np.mean(ttr_ob_bull):>8.1f} {np.median(ttr_ob_bull):>7.1f} {np.std(ttr_ob_bull):>7.1f}")
if len(ttr_ob_bear) > 0:
    print(f"    {'Overbought + Bear regime':>25} {len(ttr_ob_bear):>6} {np.mean(ttr_ob_bear):>8.1f} {np.median(ttr_ob_bear):>7.1f} {np.std(ttr_ob_bear):>7.1f}")

print("\n" + "=" * 72)
print("  END OF ANALYSIS")
print("=" * 72)